# 🚀 [ICML 2026] LiDAR: Lookahead Sample Reward Guidance for Diffusion Models
### Tái lập Bảng 2: `LiDAR (DPM-5 / n=50)` trên GenEval Benchmark (Lưu trữ vĩnh viễn trên Google Drive)

**Bài báo:** *Lookahead Sample Reward Guidance for Test-Time Scaling of Diffusion Models* ([arXiv:2602.03211](https://arxiv.org/abs/2602.03211))  
**GitHub Repository:** [github.com/leekwanreal/Noisy-Reward](https://github.com/leekwanreal/Noisy-Reward)  
**Mục tiêu đối chứng (Table 2):**
- ImageReward (IR): **0.378 ~ 0.384**
- CLIP-Score: **0.278**
- HumanPreference (HPS v2.1): **0.276 ~ 0.277**
- Time (Latency): **13.41s** (trên GPU A100)

## 📌 Step 1: Kiểm tra GPU & Kết nối Google Drive
Kết nối Google Drive để lưu toàn bộ ảnh và điểm số định lượng vĩnh viễn (không bao giờ bị mất dữ liệu khi tắt Colab).

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi

# 2. Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# 3. Thư mục lưu kết quả trên Google Drive của bạn
GDRIVE_DIR = "/content/drive/MyDrive/LiDAR_Experiment"
os.makedirs(GDRIVE_DIR, exist_ok=True)
print(f"\n✅ Đã kết nối Google Drive thành công! Thư mục lưu trữ: {GDRIVE_DIR}")

## 📦 Step 2: Cài đặt Môi trường Chuẩn & Tải Mã nguồn

In [ ]:
# 1. Tải hoặc cập nhật repo Noisy-Reward
import os
%cd /content
if not os.path.exists("/content/Noisy-Reward"):
    !git clone https://github.com/leekwanreal/Noisy-Reward.git
%cd /content/Noisy-Reward
!git pull origin main

# 2. Cài đặt các gói thư viện chuẩn
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q hpsv2 matplotlib tqdm scipy seaborn pandas

# 3. Tải bổ sung file vocab cho hpsv2 (nếu thiếu)
import urllib.request, hpsv2
hpsv2_vocab = os.path.join(os.path.dirname(hpsv2.__file__), "src", "open_clip", "bpe_simple_vocab_16e6.txt.gz")
os.makedirs(os.path.dirname(hpsv2_vocab), exist_ok=True)
if not os.path.exists(hpsv2_vocab):
    urllib.request.urlretrieve("https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", hpsv2_vocab)

print("\n✅ Môi trường đã được cài đặt hoàn tất!")

> ⚠️ **LƯU Ý:** Nếu đây là lần đầu tiên bạn vừa chạy lệnh `pip install` ở Step 2, vui lòng bấm **Runtime $\rightarrow$ Restart session** (phím tắt `Ctrl + M .`) để Colab nạp phiên bản vừa cài vào RAM, sau đó chạy tiếp các bước bên dưới!

## ⚡ Step 3: Phase 1 — Lookahead Sampling (DPM-5, n=50 particles)
- Sử dụng **Stable Diffusion v1.5** với bộ giải **DPMSolver (5 steps)**.
- Sinh **50 particles** mẫu cho mỗi prompt và đánh giá trước hàm thưởng **ImageReward**.
- Tối ưu tốc độ cao: Chỉ lưu latents và kết quả điểm số vào Google Drive, bỏ qua lưu 50 file PNG thô.
- Có cơ chế **Auto-Resume**: Nếu bị ngắt kết nối, chỉ cần bấm chạy lại cell này sẽ tự động bỏ qua các prompt cũ và chạy tiếp!

In [ ]:
%cd /content/Noisy-Reward

!python lookahead_sampling.py \
    --seed=100 \
    --model_name="runwayml/stable-diffusion-v1-5" \
    --prompt_path="prompt_files/geneval_metadata.jsonl" \
    --output_dir="/content/drive/MyDrive/LiDAR_Experiment/Lookahead_samples" \
    --num_particles=50 \
    --num_inference_steps=5 \
    --metrics_to_compute="ImageReward"

### 🧹 (Tùy chọn) Dọn dẹp ảnh PNG ở Pha 1 để giải phóng dung lượng Google Drive
Xóa các file ảnh PNG thô trong `Lookahead_samples` nhưng vẫn bảo toàn nguyên vẹn 100% các file `latent.pt` và `results.json` cho Pha 2.

In [ ]:
import glob, os

png_files = glob.glob("/content/drive/MyDrive/LiDAR_Experiment/Lookahead_samples/**/*.png", recursive=True)
print(f"🔍 Tìm thấy {len(png_files)} file ảnh PNG trong Pha 1. Đang tiến hành xóa...")
for f in png_files:
    try:
        os.remove(f)
    except Exception:
        pass
print(f"✅ Đã dọn dẹp sạch sẽ toàn bộ {len(png_files)} file ảnh PNG Pha 1!")
print("🛡️ Các file 'latent.pt' và 'results.json' vẫn được bảo toàn an toàn trên Google Drive.")

## 🎯 Step 4: Phase 2 — LiDAR Target Sampling (DDIM 50-steps / DDPM 100-steps)
- Lấy mẫu định hướng theo phân phối Target với Reward Guidance scale $w=12.5$.
- Sinh song song **4 ảnh / prompt** (chuẩn đánh giá GenEval trong Bảng 2 của bài báo).
- Toàn bộ ảnh đích lưu vào Google Drive (`/content/drive/MyDrive/LiDAR_Experiment/Target_samples`).

In [ ]:
%cd /content/Noisy-Reward

!python LiDAR_sampling.py \
    --seed=100 \
    --use_rag \
    --model_name="runwayml/stable-diffusion-v1-5" \
    --prompt_path="prompt_files/geneval_metadata.jsonl" \
    --output_dir="/content/drive/MyDrive/LiDAR_Experiment/Target_samples" \
    --num_inference_steps=50 \
    --num_particles=4 \
    --top_k=50 \
    --scale=12.5 \
    --resample_t_end=200 \
    --lookahead_path="/content/drive/MyDrive/LiDAR_Experiment/Lookahead_samples/100_50_5" \
    --metrics_to_compute="ImageReward" \
    --save_individual_images

## 📊 Step 5: Đánh giá Toàn diện (ImageReward, CLIP, HPS v2.1, Latency) & Đối chứng Bảng 2

In [ ]:
import os, glob, json, gc, torch, sys
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

# 1. Tìm thư mục kết quả mới nhất
target_runs = sorted(glob.glob("/kaggle/working/LiDAR_Experiment/Target_samples/*"))
if not target_runs:
    # Fallback tìm các thư mục khác
    target_runs = sorted(glob.glob("/content/LiDAR_Experiment/Target_samples/*") + glob.glob("Target_samples/*"))
if not target_runs:
    raise FileNotFoundError("Chưa tìm thấy thư mục kết quả. Hãy đảm bảo Step 3 đã chạy xong!")

latest_dir = target_runs[-1]
print(f"📂 Đang phân tích kết quả tại: {latest_dir}")

# 2. Thu thập danh sách ảnh và điểm ImageReward
all_images = []
all_prompts = []
ir_mean_scores = []
ir_best_scores = []
prompt_dirs = sorted([d for d in glob.glob(os.path.join(latest_dir, "*")) if os.path.isdir(d) and os.path.basename(d).isdigit()])

for p_dir in prompt_dirs:
    meta_path = os.path.join(p_dir, "metadata.jsonl")
    results_path = os.path.join(p_dir, "results.json")
    prompt_text = ""
    if os.path.exists(meta_path):
        try:
            with open(meta_path, "r", encoding="utf-8") as f:
                prompt_text = json.load(f).get("prompt", "")
        except Exception:
            pass
    if os.path.exists(results_path):
        try:
            with open(results_path, "r", encoding="utf-8") as f:
                res_data = json.load(f)
                ir_mean_scores.append(res_data.get("ImageReward", {}).get("mean", 0.0))
                ir_best_scores.append(res_data.get("ImageReward", {}).get("max", res_data.get("ImageReward", {}).get("mean", 0.0)))
        except Exception:
            pass

    # Tìm chính xác các ảnh thành phẩm trong thư mục samples/
    sample_imgs = sorted(glob.glob(os.path.join(p_dir, "samples", "*.png")))
    if not sample_imgs:
        sample_imgs = sorted([p for p in glob.glob(os.path.join(p_dir, "*.png")) if "grid" not in p])

    for img_path in sample_imgs:
        all_images.append(img_path)
        all_prompts.append(prompt_text)

ir_mean = sum(ir_mean_scores) / max(1, len(ir_mean_scores))
ir_best_mean = sum(ir_best_scores) / max(1, len(ir_best_scores))
print(f"🖼️ Tổng số ảnh sinh ra: {len(all_images)} ảnh trên {len(prompt_dirs)} prompts.")

# 3. Tính CLIP Score tuần tự
if len(all_images) > 0:
    print("\n⏳ Đang tính CLIP-Score...")
    try:
        from fks_utils import do_clip_score
    except ImportError:
        sys.path.insert(0, '/kaggle/working/Noisy-Reward/fkd_diffusers')
        from fks_utils import do_clip_score

    clip_scores = []
    for idx in tqdm(range(0, len(all_images), 10)):
        batch_imgs = [Image.open(p) for p in all_images[idx:idx+10]]
        batch_prompts = all_prompts[idx:idx+10]
        scores = do_clip_score(images=batch_imgs, prompts=batch_prompts)
        clip_scores.extend(scores)
    clip_mean = sum(clip_scores) / max(1, len(clip_scores))
else:
    clip_mean = 0.0

# 4. Tính HPS v2.1 Score tuần tự
if len(all_images) > 0:
    print("\n⏳ Đang tính HPS v2.1 Score...")
    from fkd_diffusers.rewards import do_human_preference_score
    hps_scores = []
    for idx in tqdm(range(0, len(all_images), 10)):
        batch_imgs = [Image.open(p) for p in all_images[idx:idx+10]]
        batch_prompts = all_prompts[idx:idx+10]
        scores = do_human_preference_score(images=batch_imgs, prompts=batch_prompts)
        hps_scores.extend(scores)
    hps_mean = sum(hps_scores) / max(1, len(hps_scores))
else:
    hps_mean = 0.0

# Giải phóng bộ nhớ GPU
gc.collect()
torch.cuda.empty_cache()

# 5. In bảng đối chứng toàn diện so với Bảng 2 bài báo
print("\n" + "="*78)
print("📈 KẾT QUẢ ĐỐI CHỨNG THỰC NGHIỆM VS BÀI BÁO (TABLE 2 - SD v1.5 LiDAR DPM-5/n=50)")
print("="*78)
print(f"• ImageReward (All 4 Particles Mean):  {ir_mean:.4f}  | Bài báo Table 2: 0.378 ~ 0.384")
print(f"• ImageReward (Best-of-4 Rank 1):     {ir_best_mean:.4f}  | Bài báo Table 8 BoN+LiDAR: 0.969")
print(f"• CLIP Score:                         {clip_mean:.4f}  | Bài báo Table 2: 0.278")
print(f"• HPS v2.1:                           {hps_mean:.4f}  | Bài báo Table 2: 0.272 ~ 0.275")
print("="*78)

# 6. Hiển thị ảnh mẫu
sample_grid = os.path.join(latest_dir, "00000/grid.png")
if os.path.exists(sample_grid):
    plt.figure(figsize=(16, 5))
    plt.imshow(Image.open(sample_grid))
    plt.axis("off")
    plt.title("4 Particles Generated with LiDAR (Sorted by Reward)", fontsize=14)
    plt.show()